In [1]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [2]:
email = "Hi this is a test email"

In [3]:
client = OpenAI()

prompt = f"Please classify the email {email}"

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "user", "content": prompt}
    ]
    
)

In [4]:
response

ChatCompletion(id='chatcmpl-DuSrGrPXI2lL97oIbaeyjtCwWfyui', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The email "Hi this is a test email" can be classified as a **Test Email** or **Informal Communication**. It appears to be a non-commercial message, likely intended to check functionality or communicate informally.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782349670, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_dd933fdd28', usage=CompletionUsage(completion_tokens=45, prompt_tokens=17, total_tokens=62, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [5]:
output = response.choices[0].message.content
print(output)

The email "Hi this is a test email" can be classified as a **Test Email** or **Informal Communication**. It appears to be a non-commercial message, likely intended to check functionality or communicate informally.


In [6]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The email "Hi this is a test email" can be classified as a **Test Email** or **Informal Communication**. It appears to be a non-commercial message, likely intended to check functionality or communicate informally.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))]

In [7]:
def classify_email(prompt):
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": prompt}
        ]
        
    )
    output = response.choices[0].message.content
    return output

In [8]:
import json

with open("test_emails.json", "r") as f:
    emails = json.load(f)

In [9]:
prompt = f"Please classify the email {email}"


In [10]:
emails[4]['body']

'The app crashes when I click Save. Please assist.'

In [11]:
for i in range(5):
    prompt = f"Please classify the email into oe of the categorys - Billing, Technical , Other .. email : {emails[i]['body']}"
    output = classify_email(prompt)
    print(output)


The email can be classified under the category **Billing**.
The email can be classified as **Billing**.
The email falls under the category of **Billing**.
The email can be classified as **Billing**.
This email falls into the **Technical** category.


In [13]:
email = emails[0]
email

{'id': 1,
 'subject': 'Invoice Discrepancy',
 'sender': 'finance@company.com',
 'body': 'My invoice is wrong, please fix it.',
 'correct_label': 'Billing',
 'urgency': 'High',
 'difficulty': 'easy',
 'why_tricky': 'The request is straightforward and specifically asks for a billing issue to be resolved.',
 'model_output': 'Billing'}

In [14]:

prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Email: {email['body']}"""

output = classify_email(prompt)
print(output)

Billing


In [15]:
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

EMAIL TO CLASSIFY:
Subject: {email['subject']}
From: {email['sender']}
Body: {email['body']}"""

output = classify_email(prompt)
print(output)


Category: Billing


In [16]:
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Rules:
- Return ONLY the category name. Nothing else.
- No explanation, no punctuation, no extra words.
- If unsure, return "Other".

EMAIL TO CLASSIFY:
Subject: {email['subject']}
From: {email['sender']}
Body: {email['body']}"""


output = classify_email(prompt)
print(output)

Billing


In [17]:
PROMPT_WITH_CONFIDENCE = """You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Also give a Confidence Score from 1-10:
- 10 = absolutely sure
- 5  = could go either way
- 1  = total guess

Rules:
- Return ONLY in this exact format:  Category | Score
- Example outputs:  Billing | 9
- Example outputs:  Technical | 6
- Example outputs:  Other | 3
- No explanation. Nothing else.

EMAIL TO CLASSIFY:
Subject: {subject}
From: {sender}
Body: {body}"""

In [18]:
email_input = PROMPT_WITH_CONFIDENCE.format(subject=email['subject'], sender=email['sender'], body=email['body'])
output =classify_email(email_input)
print(output)

Billing | 10


{'id': 1,
 'subject': 'Invoice Discrepancy',
 'sender': 'finance@company.com',
 'body': 'My invoice is wrong, please fix it.',
 'correct_label': 'Billing',
 'urgency': 'High',
 'difficulty': 'easy',
 'why_tricky': 'The request is straightforward and specifically asks for a billing issue to be resolved.',
 'model_output': 'Billing'}

In [22]:
category = output.split('|')[0].strip()
confidance_score = output.split('|')[1].strip()
category

'Billing'

In [25]:
# emails
email["model_output"] = category
email["confidence_score"] = confidance_score
email

{'id': 1,
 'subject': 'Invoice Discrepancy',
 'sender': 'finance@company.com',
 'body': 'My invoice is wrong, please fix it.',
 'correct_label': 'Billing',
 'urgency': 'High',
 'difficulty': 'easy',
 'why_tricky': 'The request is straightforward and specifically asks for a billing issue to be resolved.',
 'model_output': 'Billing',
 'confidence_score': '10'}

In [26]:
procceed_email = []
for email in emails[:10]:
    email_input = PROMPT_WITH_CONFIDENCE.format(subject=email['subject'], sender=email['sender'], body=email['body'])
    output =classify_email(email_input)
    # clean the output to get the category and confidence score
    category = output.split('|')[0].strip()
    confidance_score = output.split('|')[1].strip()
    email["model_output"] = category
    email["confidence_score"] = confidance_score
    procceed_email.append(email)
    
    print("-----")

-----
-----
-----
-----
-----
-----
-----
-----
-----
-----


In [28]:
import json

with open("procceed_emails.json", "w") as f:
    json.dump(procceed_email, f)

In [31]:
# some worst conditions
long_thread = """
Hi Team,

My payment failed but I was charged twice. Please refund.

Thanks,
Mike

─────────────────────────────────────
From: Support Team
Thanks for reaching out! Can you share your account ID?

─────────────────────────────────────
From: Mike
Sure, account ID is ACC-4521. Also my team cannot login 
since this morning. Please fix urgently.

─────────────────────────────────────
From: Support Team  
We are looking into it. Meanwhile can you try clearing cache?

─────────────────────────────────────
From: Mike
Tried that. Still broken. Also wanted to ask — 
can you add a bulk export feature? 
Would be really helpful for our monthly reports.

─────────────────────────────────────
From: Billing Dept
We see the double charge. Refund has been initiated.

─────────────────────────────────────
From: Mike
Great. But login still broken. And what about the export feature?
Also our API rate limits keep hitting — payroll runs tonight.
Please help urgently.

─────────────────────────────────────
[Auto-generated footer]
This email and any attachments are confidential and intended 
solely for the use of the individual or entity to whom they 
are addressed. If you have received this email in error please 
notify the system manager. This message contains confidential 
information and is intended only for the individual named.
Please do not copy or forward this email to anyone else.
Thank you for your cooperation.
─────────────────────────────────────
""" * 3

long_email = {
    "subject": "Re: Re: Re: Payment + Login + Feature",
    "sender": "mike@clientcorp.com",
    "body": long_thread,
    # "correct_label": "Billing"
} 

print("email len:",len(long_thread))
print("Approx token:",len(long_thread)//4)

email len: 4380
Approx token: 1095


In [32]:
CLASSIFY_PROMPT = """You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Rules:
- Return ONLY the category name. Nothing else.
- No explanation, no punctuation, no extra words.
- If unsure, return "Other".

EMAIL TO CLASSIFY:
Subject: {subject}
From: {sender}
Body: {body}"""

In [34]:
prompt =CLASSIFY_PROMPT.format(**long_email)

In [37]:
# print(prompt)
classify_email(prompt)

'Technical'

In [72]:
# load the prompts from the md files

import glob
prompts = []
for prompt_file in glob.glob("prompt/*.md"):
    with open(prompt_file, "r") as f:
        prompts.append(f.read())
        


In [73]:
email['body'] = "our SFDC integration is broken"

In [74]:
email

{'id': 10,
 'subject': 'Add Custom Fields',
 'sender': 'pm@business.com',
 'body': 'our SFDC integration is broken',
 'correct_label': 'Feature Request',
 'urgency': 'Medium',
 'difficulty': 'easy',
 'why_tricky': 'The request clearly asks for a feature enhancement, making it easy to classify.',
 'model_output': 'Feature Request',
 'confidence_score': '9'}

In [75]:
prompts[1]

'CRITICAL: Return ONLY valid JSON. No explanation. No markdown.\n\n# ROLE\nYou are a support email classifier for a B2B SaaS company with deep \nknowledge of our product and customer patterns.\n\n# EXAMPLES OF CORRECT CLASSIFICATIONS\nStudy these examples carefully — they reflect OUR specific business context:\nexamples = [\n        {\n            "subject": "Charged $299 instead of $99",\n            "body": "I upgraded to the Pro plan but you charged me $299, the Enterprise price.",\n            "output": {"category": "Billing", "urgency": "High", "billing_sub": "Failed Payment"}\n        },\n        {\n            "subject": "SFDC integration not syncing",\n            "body": "Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.",\n            "output": {"category": "Technical", "urgency": "High", "billing_sub": "Not Applicable"}\n        },\n        {\n            "subject": "Can you add Slack notifications?",\n            "body": "Would love to get a Sl

In [76]:
prompt = prompts[1]
output =classify_email(prompt.format(**email))

KeyError: '\n            "subject"'

In [59]:
print(output) # output

{
  "category": "Technical",
  "urgency": "High",
  "billing_sub": "Not Applicable"
}


In [77]:
prompt

'CRITICAL: Return ONLY valid JSON. No explanation. No markdown.\n\n# ROLE\nYou are a support email classifier for a B2B SaaS company with deep \nknowledge of our product and customer patterns.\n\n# EXAMPLES OF CORRECT CLASSIFICATIONS\nStudy these examples carefully — they reflect OUR specific business context:\nexamples = [\n        {\n            "subject": "Charged $299 instead of $99",\n            "body": "I upgraded to the Pro plan but you charged me $299, the Enterprise price.",\n            "output": {"category": "Billing", "urgency": "High", "billing_sub": "Failed Payment"}\n        },\n        {\n            "subject": "SFDC integration not syncing",\n            "body": "Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.",\n            "output": {"category": "Technical", "urgency": "High", "billing_sub": "Not Applicable"}\n        },\n        {\n            "subject": "Can you add Slack notifications?",\n            "body": "Would love to get a Sl

In [79]:
PRODUCT_CONFIGS = {
    "saas_crm": {
        "product_name": "CRM Platform",
        "categories": ["Billing", "Technical", "Feature Request", "Spam", "Other"],
        "urgency_levels": ["High", "Medium", "Low"],
        "sla_hours": {"High": 2, "Medium": 8, "Low": 24},
        "examples": [
            {"subject": "SFDC not syncing", "body": "Salesforce stopped pulling leads",
             "output": {"category": "Technical", "urgency": "High"}}
        ]
    },
    "fintech_payments": {
        "product_name": "Payment Gateway",
        "categories": ["Transaction Failed", "Fraud Alert", "Settlement Delay", "KYC Issue", "Other"],
        "urgency_levels": ["Critical", "High", "Medium"],
        "sla_hours": {"Critical": 1, "High": 4, "Medium": 12},
        "examples": [
            {"subject": "Payment declined at checkout", "body": "Card declined even though sufficient balance",
             "output": {"category": "Transaction Failed", "urgency": "High"}}
        ]
    }
}

In [90]:
config = PRODUCT_CONFIGS["fintech_payments"]
categories_str = "\n".join([f"  - {cat}" for cat in config["categories"]])
sla_str = ", ".join([f"{k}: {v}hr response" for k, v in config["sla_hours"].items()])
subject = email["subject"]
body = email["body"]

In [95]:
email

{'id': 10,
 'subject': 'Add Custom Fields',
 'sender': 'pm@business.com',
 'body': 'our SFDC integration is broken',
 'correct_label': 'Feature Request',
 'urgency': 'Medium',
 'difficulty': 'easy',
 'why_tricky': 'The request clearly asks for a feature enhancement, making it easy to classify.',
 'model_output': 'Feature Request',
 'confidence_score': '9'}

In [91]:
prompt = f"""CRITICAL: Return ONLY valid JSON.

# ROLE
You are an expert support email classifier for {config['product_name']}.

# CATEGORIES
Classify into exactly one of:
{categories_str}

# URGENCY LEVELS  
Use exactly one of: {', '.join(config['urgency_levels'])}
SLAs: {sla_str}


# EMAIL TO CLASSIFY
Subject: {subject}
Body: {body}

REMINDER: Return ONLY JSON with keys "category" and "urgency"."""

In [92]:
print(prompt) # prompt

CRITICAL: Return ONLY valid JSON.

# ROLE
You are an expert support email classifier for Payment Gateway.

# CATEGORIES
Classify into exactly one of:
  - Transaction Failed
  - Fraud Alert
  - Settlement Delay
  - KYC Issue
  - Other

# URGENCY LEVELS  
Use exactly one of: Critical, High, Medium
SLAs: Critical: 1hr response, High: 4hr response, Medium: 12hr response


# EMAIL TO CLASSIFY
Subject: Add Custom Fields
Body: our SFDC integration is broken

REMINDER: Return ONLY JSON with keys "category" and "urgency".


In [93]:
output = classify_email(prompt)

In [94]:
output

'{\n  "category": "Other",\n  "urgency": "Medium"\n}'